# Roman Urdu Scam / Fraud SMS Detector - Dataset Exploration**Partner A - Data & Preprocessing Lead**This notebook explores the raw dataset and documents the preprocessing results.Every number is calculated from the data, so the notebook stays correct if thedataset changes.Sections:1. Load the raw dataset2. Inspect the raw columns (before any cleaning)3. Data quality problems found4. Run the preprocessing pipeline5. Dataset statistics and charts6. Before / after preprocessing examples7. Word analysis8. Template duplication (data-leakage check)

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when the notebook runs from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from preprocessing import config
from preprocessing.data_loader import load_dataset

pd.set_option("display.max_colwidth", 90)
print("Project root:", PROJECT_ROOT)

## 1. Load the raw dataset

In [ ]:
raw = load_dataset()

print("Shape :", raw.shape)
print("Columns:", list(raw.columns))
raw.head()

## 2. Inspect the raw columns (before any cleaning)Note the inconsistent capitalisation and the trailing space in one label.`repr()` is used so hidden whitespace is visible.

In [ ]:
print("=== status (the ML target) ===")
for value, count in raw[config.LABEL_COLUMN].value_counts(dropna=False).items():
    print(f"  {value!r:<14} -> {count}")

print("\n=== Authentic/Synthetic (reporting only, NOT the target) ===")
for value, count in raw[config.SOURCE_COLUMN].value_counts(dropna=False).items():
    print(f"  {value!r:<14} -> {count}")

## 3. Data quality problems found`validate_dataset()` inspects the data without changing it.

In [ ]:
from preprocessing.validator import validate_dataset

validation = validate_dataset(raw)
pd.Series(validation).to_frame("value")

In [ ]:
# The whitespace-only message, and the message that carries two different labels.
blank = raw[raw[config.TEXT_COLUMN].astype(str).str.strip() == ""]
print(f"Whitespace-only messages: {len(blank)}")
display(blank)

key = raw[config.TEXT_COLUMN].astype(str).str.strip().str.lower()
labels_per_message = raw.assign(_k=key).groupby("_k")[config.LABEL_COLUMN].nunique()
conflicting = set(labels_per_message[labels_per_message > 1].index)
print(f"\nMessages with contradictory labels: {len(conflicting)}")
display(raw[key.isin(conflicting)])

## 4. Run the preprocessing pipelineThis performs every step: validation, label normalization, duplicate removal,text cleaning, Roman Urdu normalization, stopword removal, the 80/20 split andTF-IDF feature extraction.

In [ ]:
from preprocessing import run_preprocessing_pipeline

results = run_preprocessing_pipeline(save_outputs=True, show_examples=False)

cleaned = results["cleaned_dataframe"]
report = results["report"]

## 5. Dataset statistics and charts

In [ ]:
stats = report["statistics"]

summary = pd.DataFrame(
    {
        "Metric": [
            "Initial records",
            "Empty messages removed",
            "Duplicates removed",
            "Contradictory copies dropped",
            "Final records",
            "Scam",
            "Genuine",
            "Authentic",
            "Synthetic",
            "Training records",
            "Testing records",
            "TF-IDF features",
        ],
        "Value": [
            report["initial_records"],
            report["invalid_row_removal"]["empty_messages_removed"],
            report["duplicate_removal"]["duplicates_removed"],
            report["duplicate_removal"]["conflicting_rows_removed"],
            report["final_records"],
            stats["scam_count"],
            stats["genuine_count"],
            stats["authentic_count"],
            stats["synthetic_count"],
            report["split"]["training_records"],
            report["split"]["testing_records"],
            report["tfidf"]["n_features"],
        ],
    }
)
summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# --- Chart 1: Scam vs Genuine -------------------------------------------
label_counts = cleaned[config.LABEL_COLUMN].value_counts()
axes[0].bar(label_counts.index, label_counts.values, color=["#c0392b", "#27ae60"])
axes[0].set_title("Scam vs Genuine")
axes[0].set_ylabel("Number of messages")
for i, value in enumerate(label_counts.values):
    axes[0].text(i, value, str(value), ha="center", va="bottom")

# --- Chart 2: Authentic vs Synthetic ------------------------------------
source_counts = cleaned[config.SOURCE_COLUMN].value_counts()
axes[1].bar(source_counts.index, source_counts.values, color=["#2980b9", "#8e44ad"])
axes[1].set_title("Authentic vs Synthetic")
axes[1].set_ylabel("Number of messages")
for i, value in enumerate(source_counts.values):
    axes[1].text(i, value, str(value), ha="center", va="bottom")

# --- Chart 3: dataset size before / after cleaning ----------------------
sizes = [report["initial_records"], report["final_records"]]
axes[2].bar(["Before cleaning", "After cleaning"], sizes, color=["#7f8c8d", "#16a085"])
axes[2].set_title("Dataset size before / after cleaning")
axes[2].set_ylabel("Number of records")
for i, value in enumerate(sizes):
    axes[2].text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.show()

In [ ]:
# Message length before and after cleaning.
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(
    cleaned[config.TEXT_COLUMN].astype(str).str.split().str.len(),
    bins=30, alpha=0.7, label="Original", color="#7f8c8d",
)
ax.hist(
    cleaned[config.CLEANED_TEXT_COLUMN].astype(str).str.split().str.len(),
    bins=30, alpha=0.7, label="Cleaned", color="#16a085",
)
ax.set_xlabel("Words per message")
ax.set_ylabel("Number of messages")
ax.set_title("Message length before and after preprocessing")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Before / after preprocessing examples

In [ ]:
examples = cleaned.sample(8, random_state=config.RANDOM_STATE)[
    [config.TEXT_COLUMN, config.CLEANED_TEXT_COLUMN, config.LABEL_COLUMN]
]
examples.columns = ["Original SMS", "Cleaned SMS", "Status"]
examples.reset_index(drop=True)

## 7. Word analysisWhich words appear most often in each class? This is descriptive statisticsabout the data - model performance analysis belongs to Partner B.

In [ ]:
from collections import Counter

scam_words = Counter(
    " ".join(cleaned.loc[cleaned[config.LABEL_COLUMN] == "Scam", config.CLEANED_TEXT_COLUMN]).split()
)
genuine_words = Counter(
    " ".join(cleaned.loc[cleaned[config.LABEL_COLUMN] == "Genuine", config.CLEANED_TEXT_COLUMN]).split()
)

comparison = pd.DataFrame(
    {
        "Top Scam words": [w for w, _ in scam_words.most_common(15)],
        "Scam count": [c for _, c in scam_words.most_common(15)],
        "Top Genuine words": [w for w, _ in genuine_words.most_common(15)],
        "Genuine count": [c for _, c in genuine_words.most_common(15)],
    }
)
comparison

In [ ]:
# Words that are strongly associated with one class (appear >=4x more often).
scam_only = [w for w, c in scam_words.most_common(200) if genuine_words[w] * 4 < c]
genuine_only = [w for w, c in genuine_words.most_common(200) if scam_words[w] * 4 < c]

print("Strong SCAM indicators   :", scam_only[:20])
print("\nStrong GENUINE indicators:", genuine_only[:20])

## 8. Template duplication (data-leakage check)Most synthetic messages come from a small number of templates where only theamount and the link change. After cleaning, those messages become **identical**strings. If a message and its twin end up on opposite sides of the train/testsplit, the model is graded on text it has already memorized, and the accuracylooks better than it really is.The pipeline measures this so it can be reported honestly.

In [ ]:
overlap = report["train_test_overlap"]
templates = report["cleaned_text_duplicates"]

print(f"Unique cleaned texts          : {templates['unique_cleaned_texts']}")
print(f"Rows sharing a cleaned text   : {templates['cleaned_duplicate_rows']}")
print(f"Template groups               : {templates['cleaned_duplicate_groups']}")
print(f"Largest template group        : {templates['largest_template_group']} messages")
print()
print(f"Test messages                 : {overlap['test_records']}")
print(f"...also present in training   : {overlap['test_records_also_in_train']}"
      f" ({overlap['test_overlap_percent']}%)")

In [ ]:
# The biggest template group: same cleaned text, different amounts and links.
biggest = cleaned[config.CLEANED_TEXT_COLUMN].value_counts().idxmax()
group = cleaned[cleaned[config.CLEANED_TEXT_COLUMN] == biggest]

print(f"This ONE cleaned text covers {len(group)} original messages:\n")
print(f"  CLEANED: {biggest}\n")
for message in group[config.TEXT_COLUMN].head(5):
    print(f"  raw: {message}")

For a leakage-free evaluation, run the pipeline in strict mode, which also dropsduplicates of the cleaned text:```bashpython run_preprocessing.py --strict```That reduces the dataset to the unique cleaned messages and brings thetrain/test overlap to **0%**. Reporting both numbers - the full-dataset resultand the strict result - is the most honest way to present this project.

## 9. Handover to Partner BThe pipeline returns everything Partner B needs:```pythonX_train_tfidf    = results["X_train_tfidf"]X_test_tfidf     = results["X_test_tfidf"]y_train          = results["y_train"]y_test           = results["y_test"]tfidf_vectorizer = results["tfidf_vectorizer"]```

In [ ]:
print("X_train_tfidf   :", results["X_train_tfidf"].shape)
print("X_test_tfidf    :", results["X_test_tfidf"].shape)
print("y_train         :", len(results["y_train"]), "labels")
print("y_test          :", len(results["y_test"]), "labels")
print("tfidf_vectorizer:", type(results["tfidf_vectorizer"]).__name__,
      "with", len(results["tfidf_vectorizer"].vocabulary_), "features")
print("\nSaved files:")
for name, path in report["output_files"].items():
    print(f"  {name:22s} {path}")